In [ ]:
pip install -U langchain-community

In [ ]:
!pip install wikipedia

In [ ]:
!pip install groq

In [ ]:
!pip install gradio

In [ ]:
!pip install sentencepiece langchain faiss-cpu datasets bert-score rouge-score wikipedia-api textstat

In [ ]:
# Download spacy model for tokenization
!python -m spacy download en_core_web_sm

In [ ]:
import os
import re
import torch
import numpy as np
import pandas as pd
import spacy
import nltk
import textstat
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from collections import Counter

# Transformers and Hugging Face
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    BertTokenizer, BertModel, T5ForConditionalGeneration, T5Tokenizer,
    pipeline, BartForConditionalGeneration, BartTokenizer
)

# Evaluation metrics
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# For LangChain and RAG
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader

# For UI
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import gradio as gr


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
nltk.download('punkt_tab')
# Load spaCy model
nlp = spacy.load('en_core_web_sm')
from datasets import load_dataset

In [ ]:
dataset = load_dataset("scientific_papers", "arxiv", split="train[:2%]")
print(f"Dataset loaded with {len(dataset)} examples")

# Sample Example
print("\nSample Paper:")
print("Title:", dataset[0]['article'].split('\n')[0])
print("Abstract (first 200 chars):", dataset[0]['abstract'][:200] + "...")
print("Article length (chars):", len(dataset[0]['article']))

In [3]:
def clean_text(text):
    """Basic text cleaning"""
    # Remove special LaTeX commands
    text = re.sub(r'\\.*?{.*?}', '', text)
    # Remove Math expressions (often in LaTeX format)
    text = re.sub(r'\$.*?\$', '[MATH]', text)
    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

In [ ]:
def preprocess_paper(paper, max_length=1024):
    """Process a paper to extract title, abstract, and main content"""
    lines = paper.strip().split('\n')
    title = lines[0] if lines else ""

    # Extract content, truncate if necessary
    content = ' '.join(lines[1:])
    content = clean_text(content)

    # Truncate to max_length
    if len(content) > max_length:
        content = content[:max_length]

    return {
        "title": title,
        "content": content
    }

# Apply preprocessing to the dataset
processed_dataset = dataset.map(lambda x: {
    "processed_article": clean_text(x["article"]),
    "processed_abstract": clean_text(x["abstract"])
})

print("\nPreprocessed sample:")
print("Original abstract length:", len(dataset[0]['abstract']))
print("Processed abstract length:", len(processed_dataset[0]['processed_abstract']))

In [ ]:
nltk.download('stopwords')
nltk.download('punkt_tab')

In [ ]:
# Analyze vocabulary distribution
def analyze_vocabulary(texts, n=50):
    """Analyze the vocabulary of the texts and return n most common words"""
    all_words = []
    stop_words = set(stopwords.words('english'))

    for text in texts[:500]:  # Use 500 samples for analysis
        words = word_tokenize(text.lower())
        words = [word for word in words if word.isalnum() and word not in stop_words]
        all_words.extend(words)

    word_counts = Counter(all_words)
    return word_counts.most_common(n)

# Get most common words in abstracts
common_words = analyze_vocabulary(processed_dataset['processed_abstract'])
print("\nMost common words in abstracts:")
for word, count in common_words:
    print(f"{word}: {count}")

In [7]:
class TransformerSummarizer:
    """Base class for transformer-based summarization models"""

    def __init__(self, model_name="facebook/bart-large-cnn"):
        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        self.model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model")

    def summarize(self, text, max_length=150, min_length=50):
        inputs = self.tokenizer.encode("summarize: " + text,
                                       return_tensors="pt",
                                       max_length=1024,
                                       truncation=True).to(device)

        summary_ids = self.model.generate(inputs,
                                         max_length=max_length,
                                         min_length=min_length,
                                         length_penalty=2.0,
                                         num_beams=4,
                                         early_stopping=True)

        summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

In [ ]:
# Test the transformer summarizer
transformer_summarizer = TransformerSummarizer()
test_paper = processed_dataset[5]['processed_article']
test_summary = transformer_summarizer.summarize(test_paper[:1024])
print("\nTest Summary from BART:")
print(test_summary)

In [ ]:
class TextSimplifier:
    """Class for simplifying complex text using T5"""

    def __init__(self, model_name="t5-base"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model for text simplification")

    def simplify(self, text, max_length=150):
        # T5 requires a specific format for text simplification
        input_text = "simplify: " + text

        inputs = self.tokenizer.encode(input_text,
                                      return_tensors="pt",
                                      max_length=512,
                                      truncation=True).to(device)

        outputs = self.model.generate(inputs,
                                     max_length=max_length,
                                     min_length=30,
                                     length_penalty=2.0,
                                     num_beams=4,
                                     early_stopping=True)

        simplified_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return simplified_text

# Initialize the text simplifier
text_simplifier = TextSimplifier()

In [ ]:
complex_sentence = "The quantum mechanical model elucidates the probabilistic nature of electron behavior in atomic orbitals, highlighting the uncertainty principle's implications on our ability to simultaneously determine position and momentum."
simplified = text_simplifier.simplify(complex_sentence)
print("\nOriginal complex text:")
print(complex_sentence)
print("\nSimplified text:")
print(simplified)

In [ ]:
class BertExtractiveSummarizer:
    """Extractive summarization using BERT embeddings"""

    def __init__(self):
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.model = BertModel.from_pretrained('bert-base-uncased').to(device)
        print("Loaded BERT for extractive summarization")

    def summarize(self, text, num_sentences=3):
        # Split text into sentences
        sentences = sent_tokenize(text)
        if len(sentences) <= num_sentences:
            return text

        # Get embeddings for each sentence
        inputs = self.tokenizer(sentences, return_tensors='pt',
                               padding=True, truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = self.model(**inputs)
            # Use CLS token embedding as sentence representation
            embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

        # Compute similarity matrix
        similarity_matrix = np.matmul(embeddings, embeddings.T)

        # Compute centrality scores (sum of similarities)
        centrality_scores = np.sum(similarity_matrix, axis=1)

        # Get top sentences
        top_sentence_indices = np.argsort(centrality_scores)[-num_sentences:]
        top_sentence_indices = sorted(top_sentence_indices)

        # Combine sentences in original order
        summary = ' '.join([sentences[i] for i in top_sentence_indices])
        return summary

# Initialize BERT extractive summarizer
bert_extractive = BertExtractiveSummarizer()

In [ ]:
extractive_summary = bert_extractive.summarize(test_paper[:2000])
print("\nBERT Extractive Summary:")
print(extractive_summary)

In [13]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter

class KnowledgeEnhancer:
    """Enhance summaries with relevant background knowledge for beginners"""

    def __init__(self):
        # Initialize embeddings and vector store
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        self.knowledge_base = self._create_sample_knowledge_base()
        print("Initialized Knowledge Enhancer with sample knowledge base")

    def _create_sample_knowledge_base(self):
        """Create a sample knowledge base with definitions and explanations"""
        knowledge_texts = [
            "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.",
            "Neural networks are computing systems inspired by the biological neural networks in animal brains.",
            "Natural language processing (NLP) is a subfield of linguistics, computer science, and AI concerned with interactions between computers and human language.",
            "Transformer models are a type of neural network architecture that uses self-attention mechanisms.",
            "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based machine learning technique for NLP pre-training developed by Google.",
            "T5 (Text-to-Text Transfer Transformer) treats every NLP problem as a text-to-text problem.",
            "BART (Bidirectional and Auto-Regressive Transformers) is a transformer encoder-decoder model designed for sequence-to-sequence tasks.",
            "Quantum mechanics is a fundamental theory in physics that describes nature at the scale of atoms and subatomic particles.",
            "LSTM (Long Short-Term Memory) is a type of recurrent neural network capable of learning long-term dependencies.",
            "RNN (Recurrent Neural Network) is a class of neural networks where connections between nodes form a directed graph along a temporal sequence."
        ]

        # Split texts into chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
        knowledge_docs = [{"content": text, "id": i} for i, text in enumerate(knowledge_texts)]

        # Create vector store
        vector_store = FAISS.from_texts(
            texts=[doc["content"] for doc in knowledge_docs],
            embedding=self.embeddings,
            metadatas=knowledge_docs
        )

        return vector_store

    def enhance_summary(self, summary, original_text, num_contexts=2):
        """Add relevant background knowledge to the summary for beginners"""
        # Extract key terms or concepts that might need explanation
        doc = nlp(original_text)

        # Extract entities and noun chunks as potential terms to explain
        key_terms = set()
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PRODUCT", "EVENT", "LAW", "WORK_OF_ART"]:
                key_terms.add(ent.text.lower())

        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) <= 3:  # Only short noun phrases
                key_terms.add(chunk.text.lower())

        # Get the most relevant knowledge for the summary
        retrieved_docs = self.knowledge_base.similarity_search(summary, k=num_contexts)

        # Add explanations to the summary
        enhanced_summary = summary + "\n\nAdditional explanations for beginners:\n"
        for i, doc in enumerate(retrieved_docs):
            enhanced_summary += f"{i+1}. {doc.page_content}\n"

        return enhanced_summary


In [ ]:
knowledge_enhancer = KnowledgeEnhancer()

# Test knowledge enhancement
beginner_summary = transformer_summarizer.summarize(test_paper[:1024], max_length=100)
enhanced_summary = knowledge_enhancer.enhance_summary(beginner_summary, test_paper[:1024])

print("\nRegular Summary:")
print(beginner_summary)
print("\nKnowledge-Enhanced Summary for Beginners:")
print(enhanced_summary)

In [ ]:
class AdaptiveSummarizer:
    """Main system that generates summaries based on reader expertise level"""

    def __init__(self):
        self.transformer_summarizer = TransformerSummarizer()
        self.text_simplifier = TextSimplifier()
        self.bert_extractive = BertExtractiveSummarizer()
        self.knowledge_enhancer = KnowledgeEnhancer()
        print("Initialized Adaptive Summarizer System")

    def summarize(self, text, expertise_level="Intermediate", max_length=None):
        """
        Parameters:
        - text: The scientific paper text to summarize
        - expertise_level: One of "Beginner", "Intermediate", "Expert"
        - max_length: Maximum length of the summary

        Returns:
        - Summary text customized to the expertise level
        """
        if max_length is None:
            max_length = 150 if expertise_level == "Expert" else 200

        # Step 1: Generate base summary using transformer
        if expertise_level == "Expert":
            # For experts: Use extractive summarization to focus on key technical details
            base_summary = self.bert_extractive.summarize(text, num_sentences=5)

            # Add a section with key findings and contributions
            key_findings = self.transformer_summarizer.summarize(
                text, max_length=100, min_length=50
            )

            final_summary = f"Technical Summary:\n{base_summary}\n\nKey Contributions:\n{key_findings}"

        elif expertise_level == "Intermediate":
            # For intermediate: Standard abstractive summary
            base_summary = self.transformer_summarizer.summarize(
                text, max_length=max_length, min_length=min(50, max_length // 2)
            )
            final_summary = base_summary

        else:  # Beginner
            # For beginners: Simplified language + additional context
            # Get a slightly shorter base summary to make room for explanations
            base_summary = self.transformer_summarizer.summarize(
                text, max_length=max_length // 2, min_length=min(30, max_length // 3)
            )

            # Simplify the language
            simplified_summary = self.text_simplifier.simplify(base_summary)

            # Add relevant background knowledge
            final_summary = self.knowledge_enhancer.enhance_summary(
                simplified_summary, text, num_contexts=3
            )

        return final_summary

    def analyze_readability(self, text):
        """Analyze readability metrics of a text"""
        metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(text),
            "SMOG Index": textstat.smog_index(text),
            "Coleman-Liau Index": textstat.coleman_liau_index(text),
            "Automated Readability": textstat.automated_readability_index(text),
            "Dale-Chall Readability": textstat.dale_chall_readability_score(text)
        }
        return metrics


# Initialize the adaptive summarizer
adaptive_summarizer = AdaptiveSummarizer()


In [16]:
def evaluate_summary(generated_summary, reference_summary):
    """Evaluate a generated summary against a reference summary"""
    # Initialize result dictionary
    result = {}

    # ROUGE scores
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores = scorer.score(reference_summary, generated_summary)

    result["ROUGE-1"] = rouge_scores["rouge1"].fmeasure
    result["ROUGE-2"] = rouge_scores["rouge2"].fmeasure
    result["ROUGE-L"] = rouge_scores["rougeL"].fmeasure

    # BLEU score with smoothing
    smoothie = SmoothingFunction().method1
    bleu_score = sentence_bleu(
        [reference_summary.split()],
        generated_summary.split(),
        smoothing_function=smoothie
    )
    result["BLEU"] = bleu_score

    # BERTScore
    P, R, F1 = bert_score([generated_summary], [reference_summary], lang='en', rescale_with_baseline=True)
    result["BERTScore"] = F1.item()

    # Readability metrics
    result["Flesch-Kincaid Grade"] = textstat.flesch_kincaid_grade(generated_summary)
    result["Dale-Chall Score"] = textstat.dale_chall_readability_score(generated_summary)

    return result

In [ ]:
# Test evaluation metrics on sample summaries
test_reference = processed_dataset[10]['processed_abstract']
test_paper_content = processed_dataset[10]['processed_article'][:2000]

summaries = {
    "Beginner": adaptive_summarizer.summarize(test_paper_content, "Beginner"),
    "Intermediate": adaptive_summarizer.summarize(test_paper_content, "Intermediate"),
    "Expert": adaptive_summarizer.summarize(test_paper_content, "Expert")
}

# Calculate metrics
evaluation_results = {}
for level, summary in summaries.items():
    evaluation_results[level] = evaluate_summary(summary, test_reference)

# Print evaluation results
print("\nEvaluation Results:")
for level, metrics in evaluation_results.items():
    print(f"\n--- {level} Level Summary ---")
    print(f"Summary: {summaries[level][:150]}...")
    for metric, value in metrics.items():
        print(f"{metric}: {value:.4f}")

In [ ]:
def plot_metrics(metrics_dict):
    """Plot evaluation metrics for different expertise levels"""
    # Prepare data
    levels = list(metrics_dict.keys())
    metrics = list(metrics_dict[levels[0]].keys())

    # Set up figure
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    # Plot each metric
    for i, metric in enumerate(metrics[:6]):  # Plot first 6 metrics
        values = [metrics_dict[level][metric] for level in levels]
        axes[i].bar(levels, values, color=['green', 'blue', 'red'])
        axes[i].set_title(metric)
        axes[i].set_ylim(0, max(values) * 1.2)  # Add some space above bars

        # Add values on top of bars
        for j, value in enumerate(values):
            axes[i].text(j, value + (max(values) * 0.05), f'{value:.3f}',
                        ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig('metrics_comparison.png')
    plt.show()

# Plot evaluation results
plot_metrics(evaluation_results)

In [ ]:
def create_interactive_demo():
    """Create an interactive demo using ipywidgets"""
    # Create widgets
    input_box = widgets.Textarea(
        placeholder='Paste your scientific paper here...',
        description='Input:',
        layout=widgets.Layout(width='100%', height='200px')
    )

    expertise_dropdown = widgets.Dropdown(
        options=['Beginner', 'Intermediate', 'Expert'],
        value='Intermediate',
        description='Reader Expertise:'
    )

    max_length_slider = widgets.IntSlider(
        value=200,
        min=100,
        max=500,
        step=50,
        description='Max Length:',
        disabled=False
    )

    summarize_button = widgets.Button(
        description='Generate Summary',
        button_style='success',
        tooltip='Click to generate a summary based on expertise level'
    )

    output_area = widgets.Output()
    metrics_output = widgets.Output()

    # Define button click handler
    def on_button_clicked(b):
        with output_area:
            clear_output()
            text = input_box.value

            if not text or len(text) < 100:
                print("Please enter a longer scientific text (at least 100 characters).")
                return

            expertise = expertise_dropdown.value
            max_length = max_length_slider.value

            print(f"Generating {expertise.lower()}-level summary...")
            summary = adaptive_summarizer.summarize(text, expertise, max_length)

            print("\n--- Summary ---\n")
            print(summary)

        with metrics_output:
            clear_output()
            readability_metrics = adaptive_summarizer.analyze_readability(summary)

            print("--- Readability Metrics ---")
            for metric, value in readability_metrics.items():
                print(f"{metric}: {value:.2f}")

    # Connect button to handler
    summarize_button.on_click(on_button_clicked)

    # Layout the UI
    input_section = widgets.VBox([
        widgets.HTML("<h3>Adaptive Scientific Paper Summarization</h3>"),
        input_box
    ])

    controls = widgets.HBox([
        expertise_dropdown,
        max_length_slider,
        summarize_button
    ])

    output_section = widgets.VBox([
        widgets.HTML("<h4>Generated Summary</h4>"),
        output_area,
        widgets.HTML("<h4>Metrics</h4>"),
        metrics_output
    ])

    # Return the final UI
    return widgets.VBox([input_section, controls, output_section])

# Create and display the interactive demo
interactive_demo = create_interactive_demo()
display(interactive_demo)


In [ ]:
def create_gradio_interface():
    """Create a Gradio web interface for the adaptive summarizer"""

    def summarize_paper(paper_text, expertise_level, max_length):
        if len(paper_text) < 100:
            return "Please enter a longer scientific text (at least 100 characters)."

        summary = adaptive_summarizer.summarize(
            paper_text, expertise_level, int(max_length)
        )

        readability_metrics = adaptive_summarizer.analyze_readability(summary)
        metrics_text = "\n\n--- Readability Metrics ---\n"
        for metric, value in readability_metrics.items():
            metrics_text += f"{metric}: {value:.2f}\n"

        return summary + metrics_text

    # Define the interface
    iface = gr.Interface(
        fn=summarize_paper,
        inputs=[
            gr.Textbox(lines=10, placeholder="Paste scientific paper text here...", label="Paper Text"),
            gr.Radio(["Beginner", "Intermediate", "Expert"], label="Reader Expertise", value="Intermediate"),
            gr.Slider(100, 500, value=200, step=50, label="Maximum Summary Length")
        ],
        outputs=gr.Textbox(label="Generated Summary"),
        title="Adaptive Scientific Paper Summarizer",
        description="This tool generates summaries of scientific papers customized to different levels of expertise.",
        examples=[
            [processed_dataset[15]["processed_article"][:2000], "Beginner", 200],
            [processed_dataset[20]["processed_article"][:2000], "Intermediate", 200],
            [processed_dataset[25]["processed_article"][:2000], "Expert", 200]
        ]
    )

    return iface

# Create the Gradio interface
gradio_interface = create_gradio_interface()

gradio_interface.launch(share=True)

In [ ]:
def demonstrate_full_system():
    """End-to-end demonstration of the complete system"""
    print("==== Adaptive Summarization System Demonstration ====\n")

    # Select a test paper
    test_index = 30
    paper = processed_dataset[test_index]['processed_article'][:3000]
    reference = processed_dataset[test_index]['processed_abstract']

    print(f"Original Paper (first 300 chars):\n{paper[:300]}...\n")
    print(f"Original Abstract:\n{reference}\n")

    # Generate summaries for each expertise level
    print("Generating summaries for different expertise levels...")
    summaries = {}
    for level in ["Beginner", "Intermediate", "Expert"]:
        summary = adaptive_summarizer.summarize(paper, level)
        summaries[level] = summary

        print(f"\n--- {level} Level Summary ---\n")
        print(summary)

        # Calculate metrics
        metrics = evaluate_summary(summary, reference)
        print("\nMetrics:")
        for metric, value in metrics.items():
            print(f"{metric}: {value:.4f}")

    # Compare readability scores
    print("\n--- Readability Comparison ---")
    readability_scores = {}
    for level, summary in summaries.items():
        fk_grade = textstat.flesch_kincaid_grade(summary)
        reading_ease = textstat.flesch_reading_ease(summary)
        readability_scores[level] = {"FK Grade": fk_grade, "Reading Ease": reading_ease}
        print(f"{level}: FK Grade = {fk_grade:.2f}, Reading Ease = {reading_ease:.2f}")

    # Plot readability comparison
    levels = list(readability_scores.keys())

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # FK Grade (lower is easier to read)
    fk_values = [readability_scores[level]["FK Grade"] for level in levels]
    ax1.bar(levels, fk_values, color=['green', 'blue', 'red'])
    ax1.set_title("Flesch-Kincaid Grade Level")
    ax1.set_ylabel("Grade Level")

    # Reading Ease (higher is easier to read)
    re_values = [readability_scores[level]["Reading Ease"] for level in levels]
    ax2.bar(levels, re_values, color=['green', 'blue', 'red'])
    ax2.set_title("Flesch Reading Ease")
    ax2.set_ylabel("Score")

    plt.tight_layout()
    plt.savefig('readability_comparison.png')
    plt.show()

    return summaries

# Run the full demonstration
demo_summaries = demonstrate_full_system()

In [22]:
import wikipedia
from langchain_community.document_loaders import WikipediaLoader
from langchain.retrievers import BM25Retriever
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
import requests
from tenacity import retry, stop_after_attempt, wait_exponential
import json

In [23]:
class EnhancedKnowledgeEnhancer:
    """Enhanced RAG system with Wikipedia integration and expanded knowledge base"""

    def __init__(self):
        # Initialize embeddings and vector store
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        # Create both the sample knowledge base and Wikipedia capability
        self.knowledge_base = self._create_enhanced_knowledge_base()
        print("Initialized Enhanced Knowledge Enhancer with Wikipedia integration")

        # Initialize Wikipedia API for dynamic retrieval
        wikipedia.set_lang("en")

    def _create_enhanced_knowledge_base(self):
        """Create an expanded knowledge base with definitions and explanations"""
        knowledge_texts = [
            "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data.",
            "Neural networks are computing systems inspired by the biological neural networks in animal brains.",
            "Natural language processing (NLP) is a subfield of linguistics, computer science, and AI concerned with interactions between computers and human language.",
            "Transformer models are a type of neural network architecture that uses self-attention mechanisms.",
            "BERT (Bidirectional Encoder Representations from Transformers) is a transformer-based machine learning technique for NLP pre-training developed by Google.",
            "T5 (Text-to-Text Transfer Transformer) treats every NLP problem as a text-to-text problem.",
            "BART (Bidirectional and Auto-Regressive Transformers) is a transformer encoder-decoder model designed for sequence-to-sequence tasks.",
            "Quantum mechanics is a fundamental theory in physics that describes nature at the scale of atoms and subatomic particles.",
            "LSTM (Long Short-Term Memory) is a type of recurrent neural network capable of learning long-term dependencies.",
            "RNN (Recurrent Neural Network) is a class of neural networks where connections between nodes form a directed graph along a temporal sequence.",
            "Reinforcement learning is an area of machine learning concerned with how intelligent agents ought to take actions in an environment to maximize cumulative reward.",
            "Computer vision is a field of artificial intelligence that trains computers to interpret and understand the visual world.",
            "Deep learning is part of a broader family of machine learning methods based on artificial neural networks with representation learning.",
            "Unsupervised learning is a type of machine learning where models are trained using data that is neither classified nor labeled.",
            "Transfer learning is a machine learning technique where a model developed for one task is reused as the starting point for a model on a second task.",
            "Attention mechanisms in neural networks allow the model to focus on specific parts of the input sequence when generating an output.",
            "Self-attention, also known as intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence.",
            "Fine-tuning is the process of taking a pre-trained model and further training it on a specific dataset for a particular task.",
            "A language model is a probability distribution over sequences of words or tokens that is used to generate text or predict the next word in a sequence."
        ]

        # Split texts into chunks
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
        knowledge_docs = [{"content": text, "id": i} for i, text in enumerate(knowledge_texts)]

        # Create vector store
        vector_store = FAISS.from_texts(
            texts=[doc["content"] for doc in knowledge_docs],
            embedding=self.embeddings,
            metadatas=knowledge_docs
        )

        return vector_store

    def _extract_key_terms(self, text, max_terms=5):
        """Extract important technical terms from the text"""
        doc = nlp(text)

        # Extract entities and noun chunks as potential terms to explain
        key_terms = []
        term_scores = {}

        # Collect named entities
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PRODUCT", "EVENT", "LAW", "WORK_OF_ART"]:
                term = ent.text.lower()
                if term not in term_scores:
                    term_scores[term] = 1
                else:
                    term_scores[term] += 1

        # Collect noun chunks (with preference for technical terms)
        for chunk in doc.noun_chunks:
            if 1 <= len(chunk.text.split()) <= 3:  # Only reasonable length noun phrases
                term = chunk.text.lower()
                # Higher score for terms with adjectives that could be technical
                if any(token.pos_ == "ADJ" for token in chunk):
                    term_scores[term] = term_scores.get(term, 0) + 1.5
                else:
                    term_scores[term] = term_scores.get(term, 0) + 0.8

        # Sort terms by score and get top terms
        sorted_terms = sorted(term_scores.items(), key=lambda x: x[1], reverse=True)
        return [term for term, score in sorted_terms[:max_terms]]

    def _get_wikipedia_content(self, term, max_sentences=3):
        """Retrieve content from Wikipedia for a specific term"""
        try:
            # Search for the term on Wikipedia
            search_results = wikipedia.search(term, results=1)
            if not search_results:
                return None

            # Get the page
            page = wikipedia.page(search_results[0], auto_suggest=False)

            # Get the summary and truncate to max_sentences
            summary = page.summary
            sentences = sent_tokenize(summary)
            return ' '.join(sentences[:max_sentences])

        except (wikipedia.exceptions.DisambiguationError, wikipedia.exceptions.PageError) as e:
            print(f"Wikipedia error for term '{term}': {str(e)}")
            return None
        except Exception as e:
            print(f"Error retrieving Wikipedia content for '{term}': {str(e)}")
            return None

    def enhance_summary(self, summary, original_text, num_contexts=3):
        """Add relevant background knowledge to the summary for beginners
        using both local knowledge base and Wikipedia"""

        # Extract key terms that might need explanation
        key_terms = self._extract_key_terms(original_text)

        # Get the most relevant knowledge from local knowledge base
        retrieved_docs = self.knowledge_base.similarity_search(summary, k=num_contexts)

        # Add explanations to the summary
        enhanced_summary = summary + "\n\nAdditional explanations for beginners:\n"

        # Add retrieved knowledge
        for i, doc in enumerate(retrieved_docs):
            enhanced_summary += f"{i+1}. {doc.page_content}\n"

        # Add Wikipedia explanations for the key terms
        wiki_explanations = []
        for term in key_terms:
            wiki_content = self._get_wikipedia_content(term)
            if wiki_content:
                wiki_explanations.append(f"* {term.capitalize()}: {wiki_content}")

        if wiki_explanations:
            enhanced_summary += "\n\nKey concepts from this paper:\n"
            enhanced_summary += "\n\n".join(wiki_explanations)

        return enhanced_summary

In [24]:
class KnowledgeGraphEnhancer:
    """Enhance expert summaries with knowledge graph information"""

    def __init__(self):
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-mpnet-base-v2"
        )
        print("Initialized Knowledge Graph Enhancer")

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _query_wikidata(self, entity_name):
        """Query Wikidata for information about an entity"""
        # SPARQL endpoint for Wikidata
        endpoint_url = "https://query.wikidata.org/sparql"

        # SPARQL query to get information about the entity
        query = f"""
        SELECT ?item ?itemLabel ?itemDescription ?field ?fieldLabel
        WHERE {{
            ?item rdfs:label "{entity_name}"@en.
            OPTIONAL {{ ?item wdt:P101 ?field. }}  # Field of work
            SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
        }}
        LIMIT 5
        """

        try:
            # Send request
            r = requests.get(endpoint_url, params={'format': 'json', 'query': query})
            data = r.json()

            # Process results
            results = data.get('results', {}).get('bindings', [])
            if not results:
                return None

            # Extract information
            entity_info = {
                "description": results[0].get('itemDescription', {}).get('value', ''),
                "fields": []
            }

            # Add fields of work if available
            for result in results:
                if 'fieldLabel' in result:
                    field = result['fieldLabel']['value']
                    if field not in entity_info["fields"]:
                        entity_info["fields"].append(field)

            return entity_info

        except Exception as e:
            print(f"Error querying Wikidata: {str(e)}")
            return None

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _query_semantic_scholar(self, query):
        """Query Semantic Scholar API for relevant papers"""
        try:
            url = f"https://api.semanticscholar.org/graph/v1/paper/search?query={query}&limit=3&fields=title,abstract,year,authors,url"


            response = requests.get(url)
            if response.status_code == 200:
                data = response.json()
                papers = data.get('data', [])
                return papers
            else:
                print(f"Error querying Semantic Scholar: {response.status_code}")
                return []

        except Exception as e:
            print(f"Error in Semantic Scholar query: {str(e)}")
            return []

    def enhance_expert_summary(self, summary, original_text):
        """Enhance expert summary with knowledge graph information"""
        # Extract important entities for KG lookup
        doc = nlp(original_text)
        key_entities = set()

        # Extract meaningful entities
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PERSON", "WORK_OF_ART", "EVENT", "LAW"]:
                key_entities.add(ent.text)

        # Add complex noun phrases as potential research concepts
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) >= 2 and len(chunk.text.split()) <= 4:
                # These might be research concepts
                key_entities.add(chunk.text)

        # Keep top entities by frequency
        entity_counter = Counter([e.lower() for e in key_entities])
        top_entities = [e for e, _ in entity_counter.most_common(3)]

        # Query knowledge graph for these entities
        kg_info = ""
        for entity in top_entities:
            # Query Wikidata
            entity_info = self._query_wikidata(entity)
            if entity_info:
                kg_info += f"\n\n### {entity.capitalize()}\n"
                kg_info += f"Description: {entity_info['description']}\n"
                if entity_info['fields']:
                    kg_info += f"Fields: {', '.join(entity_info['fields'])}\n"

            # Query Semantic Scholar for related papers
            papers = self._query_semantic_scholar(entity)
            if papers:
                kg_info += f"\nRecent research on {entity}:\n"
                for i, paper in enumerate(papers[:2], 1):
                    kg_info += f"{i}. {paper.get('title', 'Untitled')} ({paper.get('year', 'N/A')})\n"
                    authors = ", ".join([a.get('name', '') for a in paper.get('authors', [])][:3])
                    if authors:
                        kg_info += f"   Authors: {authors}\n"

        # Combine with the original summary
        if kg_info:
            enhanced_summary = summary + "\n\n## Related Knowledge Graph Information" + kg_info
            return enhanced_summary
        else:
            return summary

In [25]:
class ControlledTransformerSummarizer:
    """Enhanced summarizer that uses control tokens for different expertise levels"""

    def __init__(self, model_name="facebook/bart-large-cnn"):
        self.tokenizer = BartTokenizer.from_pretrained(model_name)
        self.model = BartForConditionalGeneration.from_pretrained(model_name).to(device)
        print(f"Loaded {model_name} model with control token capabilities")

        # Define control tokens for different expertise levels
        self.control_tokens = {
            "Beginner": "[BEGINNER]",
            "Intermediate": "[INTERMEDIATE]",
            "Expert": "[EXPERT]"
        }

    def summarize(self, text, expertise_level="Intermediate", max_length=150, min_length=50):
        """Generate summary with control tokens based on expertise level"""
        # Add control token for the desired expertise level
        control_token = self.control_tokens.get(expertise_level, self.control_tokens["Intermediate"])
        controlled_text = f"{control_token} {text}"

        # Tokenize and generate summary
        inputs = self.tokenizer.encode(
            "summarize: " + controlled_text,
            return_tensors="pt",
            max_length=1024,
            truncation=True
        ).to(device)

        # Generate with parameters appropriate for the expertise level
        if expertise_level == "Beginner":
            # For beginners: more straightforward language, higher repetition penalty
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=2.0,
                num_beams=4,
                repetition_penalty=1.3,  # Higher repetition penalty for clearer text
                early_stopping=True
            )
        elif expertise_level == "Expert":
            # For experts: allow more technical and dense content
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=1.0,  # Lower penalty to allow denser content
                num_beams=5,  # More beams for better search
                repetition_penalty=1.1,
                early_stopping=True
            )
        else:  # Intermediate
            # Default parameters
            summary_ids = self.model.generate(
                inputs,
                max_length=max_length,
                min_length=min_length,
                length_penalty=2.0,
                num_beams=4,
                early_stopping=True
            )

        # Decode and return summary
        summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        return summary

In [ ]:
import requests
import groq
import json
import os
import re
from tenacity import retry, stop_after_attempt, wait_exponential


class LLMEvaluator:
    """Evaluation system that uses LLMs to assess summary quality"""
    
    def __init__(self, api_key=None):
        # Try to get API key from parameter first, then environment variable
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        # Set API key in environment if provided
        if self.api_key:
            os.environ["GROQ_API_KEY"] = self.api_key
            
            # Initialize Groq client
            try:
                self.client = groq.Client(api_key=self.api_key)
                print("Groq client initialized successfully")
            except Exception as e:
                print(f"Error initializing Groq client: {e}")
                print("LLM evaluation will be simulated")
                self.client = None
        else:
            print("No Groq API key provided. LLM evaluation will be simulated.")
            self.client = None
    
    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _call_groq_api(self, prompt, model="llama3-70b-8192"):
        """Call the Groq API with the given prompt"""
        if self.client is None:
            # Simulate response if client is not available
            print("Simulating Groq API call (no client available)")
            return {"success": True, "response": self._simulate_llm_response()}
        
        try:
            # Call Groq API using the official client
            response = self.client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1,  # Low temperature for more consistent evaluation
                max_tokens=1000
            )
            
            return {
                "success": True,
                "response": response.choices[0].message.content
            }
        except Exception as e:
            print(f"Exception during API call: {str(e)}")
            return {"success": False, "error": str(e)}
    
    def _simulate_llm_response(self):
        """Simulate LLM response for evaluation when no API key is provided"""
        return """
        {
            "content_accuracy": 7,
            "technical_correctness": 8,
            "readability": 8,
            "appropriate_complexity": 7,
            "overall_quality": 7.5,
            "strengths": "The summary captures the main points of the original text well. It uses appropriate language for the target audience and maintains good coherence.",
            "weaknesses": "Some technical details are simplified too much, potentially losing nuance. The summary could better explain the significance of the findings.",
            "suggestions": "Consider adding a brief explanation of why this research matters and its potential applications."
        }
        """
    
    def evaluate_summary(self, original_text, summary, reference_summary=None, expertise_level="Intermediate"):
        """
        Evaluate the quality of a summary using an LLM
        
        Parameters:
        - original_text: The source text that was summarized
        - summary: The generated summary to evaluate
        - reference_summary: Optional reference/gold summary for comparison
        - expertise_level: The target expertise level of the summary
        
        Returns:
        - Dictionary with evaluation results
        """
        # Construct the prompt for the LLM
        prompt = f"""
        You are an expert evaluator of scientific summaries. Please evaluate the following summary
        of a scientific text based on the following criteria:

        1. Clarity: How clear and understandable is the summary?
        2. Accuracy: How accurately does it capture the key information from the original text?
        3. Appropriateness: How appropriate is it for a reader with {expertise_level.lower()} expertise?
        4. Overall Quality: What is the overall quality of the summary?

        For each criterion, provide a score from 1-5 (where 5 is best) and a brief explanation.

        Original Text:
        {original_text}

        Summary (for {expertise_level} level):
        {summary}
        """
        
        if reference_summary:
            prompt += f"""
            Reference summary (for comparison):
            {reference_summary}
            """
        
        prompt += """
        Provide your evaluation in JSON format with these keys:
        scores (with clarity, accuracy, appropriateness, overall) and feedback.
        """
        
        # Call the LLM
        result = self._call_groq_api(prompt)
        
        if result["success"]:
            try:
                # Extract JSON from response
                content = result["response"]
                
                # Try to find JSON block
                json_match = re.search(r'```json\n(.*?)\n```', content, re.DOTALL)
                if json_match:
                    json_str = json_match.group(1)
                else:
                    # Try without code block markers
                    json_match = re.search(r'\{.*\}', content, re.DOTALL)
                    if json_match:
                        json_str = json_match.group(0)
                    else:
                        # Fallback
                        json_str = content
                
                try:
                    evaluation = json.loads(json_str)
                    return evaluation
                except json.JSONDecodeError:
                    # Fallback to structured response if JSON parsing fails
                    return {
                        'scores': {
                            'clarity': 3.5,
                            'accuracy': 3.5,
                            'appropriateness': 3.5,
                            'overall': 3.5
                        },
                        'feedback': f"Unable to parse LLM response. Raw response: {content[:100]}..."
                    }
                    
            except Exception as e:
                print(f"Error processing LLM response: {e}")
                return {
                    'scores': {
                        'clarity': 3.0,
                        'accuracy': 3.0,
                        'appropriateness': 3.0,
                        'overall': 3.0
                    },
                    'feedback': f"Error occurred during response processing: {str(e)}"
                }
        else:
            # Handle API error
            print(f"Error in LLM evaluation: {result.get('error', 'Unknown error')}")
            return {
                'scores': {
                    'clarity': 0,
                    'accuracy': 0,
                    'appropriateness': 0,
                    'overall': 0
                },
                'feedback': f"API error: {result.get('error', 'Unknown error')}"
            }

In [ ]:
class EnhancedAdaptiveSummarizer:
    """Enhanced system that generates summaries based on reader expertise level
    with all the improvements: control tokens, RAG, KG, and LLM evaluation"""

    def __init__(self):
        # Load the enhanced components
        self.controlled_summarizer = ControlledTransformerSummarizer()
        self.text_simplifier = TextSimplifier()
        self.bert_extractive = BertExtractiveSummarizer()
        self.enhanced_knowledge = EnhancedKnowledgeEnhancer()
        self.kg_enhancer = KnowledgeGraphEnhancer()
        self.llm_evaluator = LLMEvaluator(api_key="")
        print("Initialized Enhanced Adaptive Summarizer System with all components")

    def summarize(self, text, expertise_level="Intermediate", max_length=None, evaluate=False):
        """
        Generate a summary customized to the specified expertise level

        Parameters:
        - text: The scientific paper text to summarize
        - expertise_level: One of "Beginner", "Intermediate", "Expert"
        - max_length: Maximum length of the summary
        - evaluate: Whether to perform LLM evaluation of the summary

        Returns:
        - Summary text customized to the expertise level
        - Evaluation results (if evaluate=True)
        """
        if max_length is None:
            max_length = 150 if expertise_level == "Expert" else 200

        # Step 1: Generate base summary using controlled transformer
        base_summary = self.controlled_summarizer.summarize(
            text,
            expertise_level=expertise_level,
            max_length=max_length,
            min_length=min(50, max_length // 2)
        )

        # Step 2: Apply expertise-specific enhancements
        if expertise_level == "Expert":
            # For experts: Add knowledge graph information for deeper context
            extractive_summary = self.bert_extractive.summarize(text, num_sentences=5)

            # Add key findings from the general summary
            key_findings = base_summary

            # Combine the extractive and abstractive summaries
            combined_summary = f"Technical Summary:\n{extractive_summary}\n\nKey Contributions:\n{key_findings}"

            # Enhance with knowledge graph information
            final_summary = self.kg_enhancer.enhance_expert_summary(combined_summary, text)

        elif expertise_level == "Intermediate":
            # For intermediate: Standard abstractive summary with some technical detail
            final_summary = base_summary

        else:  # Beginner
            # For beginners: Simplified language + additional context from RAG
            # Get a slightly shorter base summary to make room for explanations
            shorter_summary = self.controlled_summarizer.summarize(
                text,
                expertise_level="Beginner",
                max_length=max_length // 2,
                min_length=min(30, max_length // 3)
            )

            # Simplify the language even further
            simplified_summary = self.text_simplifier.simplify(shorter_summary)

            # Add relevant background knowledge using enhanced RAG
            final_summary = self.enhanced_knowledge.enhance_summary(
                simplified_summary, text, num_contexts=3
            )

        # Step 3: Evaluate the summary 
        if evaluate:
            evaluation = self.llm_evaluator.evaluate_summary(
                original_text=text,
                summary=final_summary,
                expertise_level=expertise_level
            )
            return final_summary, evaluation
        else:
            return final_summary

    def analyze_readability(self, text):
        """Analyze readability metrics of a text"""
        metrics = {
            "Flesch Reading Ease": textstat.flesch_reading_ease(text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(text),
            "SMOG Index": textstat.smog_index(text),
            "Coleman-Liau Index": textstat.coleman_liau_index(text),
            "Automated Readability": textstat.automated_readability_index(text),
            "Dale-Chall Readability": textstat.dale_chall_readability_score(text)
        }
        return metrics

In [ ]:
def create_enhanced_gradio_interface():
    """Create an improved Gradio web interface for the enhanced adaptive summarizer"""

    # Initialize the enhanced summarizer
    enhanced_summarizer = EnhancedAdaptiveSummarizer()

    def summarize_paper(paper_text, expertise_level, max_length, perform_evaluation):
        """Handler function for the Gradio interface"""
        if len(paper_text) < 100:
            return "Please enter a longer scientific text (at least 100 characters).", None

        try:
            # Generate summary with or without evaluation
            if perform_evaluation:
                summary, evaluation = enhanced_summarizer.summarize(
                    paper_text, expertise_level, int(max_length), evaluate=True
                )

        
                # Check the structure of evaluation to handle it correctly
                if isinstance(evaluation, dict) and 'scores' in evaluation:
                    # Format using the actual structure from LLMEvaluator
                    scores = evaluation.get('scores', {})
                    eval_text = f"""
                    ## Evaluation Results

                    **Clarity**: {scores.get('clarity', 'N/A')}/5
                    **Accuracy**: {scores.get('accuracy', 'N/A')}/5
                    **Appropriateness**: {scores.get('appropriateness', 'N/A')}/5
                    **Overall Quality**: {scores.get('overall', 'N/A')}/5

                    **Feedback**: {evaluation.get('feedback', 'No feedback provided')}
                    """
                else:
                    # Fallback for any other structure
                    eval_text = f"""
                    ## Evaluation Results

                    {str(evaluation)}
                    """

                # Add readability metrics
                readability_metrics = enhanced_summarizer.analyze_readability(summary)
                metrics_text = "\n\n## Readability Metrics\n"
                for metric, value in readability_metrics.items():
                    metrics_text += f"**{metric}**: {value:.2f}\n"

                return summary, eval_text + metrics_text
            else:
                summary = enhanced_summarizer.summarize(
                    paper_text, expertise_level, int(max_length), evaluate=False
                )

                # Just include readability metrics without LLM evaluation
                readability_metrics = enhanced_summarizer.analyze_readability(summary)
                metrics_text = "\n\n## Readability Metrics\n"
                for metric, value in readability_metrics.items():
                    metrics_text += f"**{metric}**: {value:.2f}\n"

                return summary, metrics_text

        except Exception as e:
            import traceback
            error_details = traceback.format_exc()
            return f"Error generating summary: {str(e)}\n\nDetails: {error_details}", None

    # Define the interface
    iface = gr.Interface(
        fn=summarize_paper,
        inputs=[
            gr.Textbox(
                lines=10,
                placeholder="Paste scientific paper text here...",
                label="Paper Text"
            ),
            gr.Radio(
                ["Beginner", "Intermediate", "Expert"],
                label="Reader Expertise",
                value="Intermediate"
            ),
            gr.Slider(
                100, 500,
                value=200,
                step=50,
                label="Maximum Summary Length"
            ),
            gr.Checkbox(
                label="Perform LLM-based Evaluation",
                value=False,
                info="Evaluates summary quality using LLM (may take longer)"
            )
        ],
        outputs=[
            gr.Textbox(label="Generated Summary"),
            gr.Markdown(label="Evaluation Results")
        ],
        title="Enhanced Adaptive Scientific Paper Summarizer",
        description="""This tool generates summaries of scientific papers customized to different levels of expertise.

        - **Beginner**: Simplified language with additional explanations of key concepts
        - **Intermediate**: Standard summary balancing technical accuracy and readability
        - **Expert**: Technical summary with research context and knowledge graph enhancements

        The LLM evaluation option provides detailed feedback on summary quality.""",
        examples=[
            [processed_dataset[15]["processed_article"][:2000], "Beginner", 200, False],
            [processed_dataset[20]["processed_article"][:2000], "Intermediate", 200, False],
            [processed_dataset[25]["processed_article"][:2000], "Expert", 200, True]
        ],
        allow_flagging="never"
    )

    return iface

# Create and launch the enhanced Gradio interface
enhanced_gradio_interface = create_enhanced_gradio_interface()
enhanced_gradio_interface.launch(share=True)

In [ ]:
def demonstrate_enhanced_system():
    """End-to-end demonstration of the complete enhanced system"""
    print("==== Enhanced Adaptive Summarization System Demonstration ====\n")

    # Select a test paper
    test_index = 30
    paper = processed_dataset[test_index]['processed_article'][:3000]
    reference = processed_dataset[test_index]['processed_abstract']

    print(f"Original Paper (first 300 chars):\n{paper[:300]}...\n")
    print(f"Original Abstract:\n{reference}\n")

    # Initialize the enhanced summarizer
    enhanced_summarizer = EnhancedAdaptiveSummarizer()

    # Generate summaries for each expertise level
    print("Generating summaries for different expertise levels...")
    summaries = {}
    evaluations = {}

    for level in ["Beginner", "Intermediate", "Expert"]:
        print(f"\nProcessing {level} level summary...")
        summary, evaluation = enhanced_summarizer.summarize(paper, level, evaluate=True)
        summaries[level] = summary
        evaluations[level] = evaluation

        print(f"\n--- {level} Level Summary ---\n")
        print(summary)

        print("\n--- Evaluation Results ---")
        # Access the evaluation results using the correct structure
        if isinstance(evaluation, dict) and 'scores' in evaluation:
            scores = evaluation.get('scores', {})
            print(f"Clarity: {scores.get('clarity', 'N/A')}/5")
            print(f"Accuracy: {scores.get('accuracy', 'N/A')}/5")
            print(f"Appropriateness: {scores.get('appropriateness', 'N/A')}/5")
            print(f"Overall Quality: {scores.get('overall', 'N/A')}/5")
            print(f"\nFeedback: {evaluation.get('feedback', 'No feedback provided')}")
        else:
            print("Evaluation data structure not as expected:")
            print(evaluation)

    # Compare readability scores
    print("\n--- Readability Comparison ---")
    readability_scores = {}
    for level, summary in summaries.items():
        fk_grade = textstat.flesch_kincaid_grade(summary)
        reading_ease = textstat.flesch_reading_ease(summary)
        readability_scores[level] = {"FK Grade": fk_grade, "Reading Ease": reading_ease}
        print(f"{level}: FK Grade = {fk_grade:.2f}, Reading Ease = {reading_ease:.2f}")

    # Plot readability comparison
    levels = list(readability_scores.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # FK Grade (lower is easier to read)
    fk_values = [readability_scores[level]["FK Grade"] for level in levels]
    ax1.bar(levels, fk_values, color=['green', 'blue', 'red'])
    ax1.set_title("Flesch-Kincaid Grade Level")
    ax1.set_ylabel("Grade Level")

    # Reading Ease (higher is easier to read)
    re_values = [readability_scores[level]["Reading Ease"] for level in levels]
    ax2.bar(levels, re_values, color=['green', 'blue', 'red'])
    ax2.set_title("Flesch Reading Ease")
    ax2.set_ylabel("Score")

    plt.tight_layout()
    plt.savefig('enhanced_readability_comparison.png')
    plt.show()

    # Plot LLM evaluation scores
    fig, ax = plt.subplots(figsize=(14, 7))

    # Metrics to plot 
    metrics = ["clarity", "accuracy", "appropriateness", "overall"]
    metric_labels = ["Clarity", "Accuracy", "Appropriateness", "Overall\nQuality"]

    # Get values for each level and metric
    x = np.arange(len(metrics))
    width = 0.25

    # Plot bars for each expertise level
    for i, level in enumerate(levels):
        # Extract values from the correct structure
        if isinstance(evaluations[level], dict) and 'scores' in evaluations[level]:
            scores = evaluations[level].get('scores', {})
            values = [scores.get(metric, 0) for metric in metrics]
        else:
            values = [0 for _ in metrics]  # Default if structure doesn't match
            
        ax.bar(x + (i-1)*width, values, width, label=level,
               color=["green", "blue", "red"][i])

  
    ax.set_ylabel('Score (out of 5)')  
    ax.set_title('LLM Evaluation Results by Expertise Level')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels)
    ax.legend()
    ax.set_ylim(0, 5)  

    # Add value labels on top of bars
    for i, level in enumerate(levels):
        if isinstance(evaluations[level], dict) and 'scores' in evaluations[level]:
            scores = evaluations[level].get('scores', {})
            values = [scores.get(metric, 0) for metric in metrics]
        else:
            values = [0 for _ in metrics]  # Default if structure doesn't match
            
        for j, v in enumerate(values):
            ax.text(j + (i-1)*width, v + 0.15, f'{v:.1f}',
                    ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.savefig('enhanced_evaluation_comparison.png')
    plt.show()

    return summaries, evaluations

# Run the enhanced demonstration
enhanced_summaries, enhanced_evaluations = demonstrate_enhanced_system()

In [ ]:
def run_comparative_evaluation():
    """Compare the original and enhanced summarization systems"""
    print("==== Comparative Evaluation: Original vs Enhanced Systems ====\n")

    # Set up test samples
    test_indices = [10, 20, 30, 40, 50]  # Use 5 different samples

    # Initialize both summarizers
    original_summarizer = AdaptiveSummarizer()
    enhanced_summarizer = EnhancedAdaptiveSummarizer()

    # Set up results storage
    results = {
        "Original": {level: {"ROUGE-1": [], "ROUGE-2": [], "ROUGE-L": [], "BLEU": [],
                            "BERTScore": [], "FK Grade": [], "Reading Ease": []}
                    for level in ["Beginner", "Intermediate", "Expert"]},
        "Enhanced": {level: {"ROUGE-1": [], "ROUGE-2": [], "ROUGE-L": [], "BLEU": [],
                            "BERTScore": [], "FK Grade": [], "Reading Ease": [],
                            "LLM Overall": []}
                    for level in ["Beginner", "Intermediate", "Expert"]}
    }

    # Process each test sample
    for i, idx in enumerate(test_indices):
        print(f"\nProcessing test sample {i+1}/{len(test_indices)} (dataset index {idx})...")

        # Get the test paper and reference
        paper = processed_dataset[idx]['processed_article'][:3000]
        reference = processed_dataset[idx]['processed_abstract']

        # Process each expertise level
        for level in ["Beginner", "Intermediate", "Expert"]:
            print(f"  Generating {level} summaries...")

            # Generate summaries with both systems
            original_summary = original_summarizer.summarize(paper, level)
            enhanced_summary, evaluation = enhanced_summarizer.summarize(paper, level, evaluate=True)

            # Evaluate original summary
            orig_metrics = evaluate_summary(original_summary, reference)

            # Evaluate enhanced summary
            enhanced_metrics = evaluate_summary(enhanced_summary, reference)

            # Store results
            for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore"]:
                results["Original"][level][metric].append(orig_metrics[metric])
                results["Enhanced"][level][metric].append(enhanced_metrics[metric])

            # Add readability metrics
            results["Original"][level]["FK Grade"].append(
                textstat.flesch_kincaid_grade(original_summary))
            results["Original"][level]["Reading Ease"].append(
                textstat.flesch_reading_ease(original_summary))

            results["Enhanced"][level]["FK Grade"].append(
                textstat.flesch_kincaid_grade(enhanced_summary))
            results["Enhanced"][level]["Reading Ease"].append(
                textstat.flesch_reading_ease(enhanced_summary))

            # Add LLM evaluation score for enhanced version - FIXED to use correct structure
            if isinstance(evaluation, dict) and 'scores' in evaluation:
                # Extract overall score from the correct path
                llm_overall_score = evaluation['scores'].get('overall', 0)
            else:
                # Default if structure doesn't match
                llm_overall_score = 0
                
            results["Enhanced"][level]["LLM Overall"].append(llm_overall_score)

    # Calculate averages
    averages = {
        "Original": {level: {} for level in ["Beginner", "Intermediate", "Expert"]},
        "Enhanced": {level: {} for level in ["Beginner", "Intermediate", "Expert"]}
    }

    for system in ["Original", "Enhanced"]:
        for level in ["Beginner", "Intermediate", "Expert"]:
            for metric, values in results[system][level].items():
                averages[system][level][metric] = sum(values) / len(values)

    # Print average results
    print("\n==== Average Evaluation Results ====")
    for level in ["Beginner", "Intermediate", "Expert"]:
        print(f"\n--- {level} Level ---")

        # Print comparison table
        print(f"{'Metric':<20} {'Original':<10} {'Enhanced':<10} {'Diff':<10}")
        print("-" * 50)

        for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU", "BERTScore",
                       "FK Grade", "Reading Ease"]:
            orig_val = averages["Original"][level][metric]
            enhanced_val = averages["Enhanced"][level][metric]
            diff = enhanced_val - orig_val

            # For readability metrics, interpret differently
            if metric == "FK Grade":
                # Lower grade level is better for Beginner
                if level == "Beginner":
                    better = "+" if diff < 0 else "-"
                # Higher grade level might be better for Expert
                elif level == "Expert":
                    better = "+" if diff > 0 else "-"
                else:
                    better = "+" if abs(diff) < 1 else "-"  # Minimal change for Intermediate
            elif metric == "Reading Ease":
                # Higher reading ease is better for Beginner
                if level == "Beginner":
                    better = "+" if diff > 0 else "-"
                # Lower reading ease might be appropriate for Expert
                elif level == "Expert":
                    better = "+" if diff < 0 else "-"
                else:
                    better = "+" if abs(diff) < 5 else "-"  # Minimal change for Intermediate
            else:
                # For all other metrics, higher is better
                better = "+" if diff > 0 else "-"

            print(f"{metric:<20} {orig_val:.4f}     {enhanced_val:.4f}     {diff:.4f} {better}")

        # Print LLM overall score for enhanced version 
        if "LLM Overall" in averages["Enhanced"][level]:
            print(f"\nLLM Overall Quality Score: {averages['Enhanced'][level]['LLM Overall']:.2f}/5")

    # Plot comparative results
    plot_comparative_results(averages)

    return results, averages

def plot_comparative_results(averages):
    """Plot comparative results between original and enhanced systems"""
    # Set up metrics to plot
    metrics_to_plot = ["ROUGE-1", "ROUGE-L", "BERTScore", "Reading Ease"]
    levels = ["Beginner", "Intermediate", "Expert"]

    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    axes = axes.flatten()

    # Plot each metric
    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]

        # Set up data
        x = np.arange(len(levels))
        width = 0.35

        # Get values
        orig_values = [averages["Original"][level][metric] for level in levels]
        enhanced_values = [averages["Enhanced"][level][metric] for level in levels]

        # Create bars
        rects1 = ax.bar(x - width/2, orig_values, width, label='Original')
        rects2 = ax.bar(x + width/2, enhanced_values, width, label='Enhanced')

        # Add labels and title
        ax.set_ylabel('Score')
        ax.set_title(f'{metric} Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(levels)
        ax.legend()

        # Add value labels
        def autolabel(rects):
            for rect in rects:
                height = rect.get_height()
                ax.annotate(f'{height:.3f}',
                            xy=(rect.get_x() + rect.get_width() / 2, height),
                            xytext=(0, 3),  # 3 points vertical offset
                            textcoords="offset points",
                            ha='center', va='bottom', fontsize=8)

        autolabel(rects1)
        autolabel(rects2)

    plt.tight_layout()
    plt.savefig('comparative_results.png')
    plt.show()

    # Create a separate plot for LLM evaluation scores 

    # Set up data
    llm_scores = [averages["Enhanced"][level]["LLM Overall"] for level in levels]

    # Create bars with custom colors
    bars = plt.bar(levels, llm_scores, color=['green', 'blue', 'red'])

    # Add labels and title 
    plt.ylabel('LLM Overall Quality Score (out of 5)')
    plt.title('LLM Evaluation of Enhanced Summaries')
    plt.ylim(0, 5) 

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        plt.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom')

    plt.savefig('llm_evaluation_scores.png')
    plt.show()

# Run the comparative evaluation
comparative_results, comparative_averages = run_comparative_evaluation()